## ENEM: KNN


### 0. Imports

In [3]:
import numpy as np
import matplotlib.pyplot as plt

from statstech.neighbors import KNeighborsRegressor
from statstech.model_selection import GridSearchCV, KFold

from analysis.config import SEED
from analysis.io import load_dataset
from analysis.models import ScaledRegressor
from analysis.report import save_figure, save_table

ModuleNotFoundError: No module named 'statstech'

### 1. Dados

In [ ]:
d = load_dataset("enem")

X_train = d["X_train"]
y_train = d["y_train"]

X_test = d["X_test"]
y_test = d["y_test"]

feature_names = d["feature_names"]

print(f"Dados de treino: {X_train.shape[0]} participantes, {X_train.shape[1]} colunas")


### 2. Validação cruzada

In [ ]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

### 3. Seleção de uma amostra de treinamento

In [ ]:
sample_size = 20000

rng = np.random.default_rng(SEED)

if len(X_train) > sample_size:
    indices = rng.choice(
        len(X_train),
        size=sample_size,
        replace=False
    )

    X_knn = X_train[indices]
    y_knn = y_train[indices]

    print("Busca utilizando uma subamostra de 20 mil observações")

else:
    X_knn = X_train
    y_knn = y_train

    print("Busca utilizando todo o conjunto de treino")

print("Dimensão da amostra:", X_knn.shape)

### 4. Construção dos modelos KNN

In [ ]:
models_knn = [
    KNeighborsRegressor(n_neighbors=k)
    for k in [5, 15, 31, 61, 121]
]

knn = GridSearchCV(
    ScaledRegressor(block_size=1000),
    {"model": models_knn},
    cv=cv,
    scoring="rmse"
)

knn.fit(X_knn, y_knn)

### 5. Identificação dos resultados

In [ ]:
best_model = knn.best_estimator_

best_k =knn.best_params_["model"].n_neighbors

best_rmse = knn.best_score_

print("Melhor valor de K:", best_k)
print("Melhor RMSE médio: {best_rmse:.4f}")

### 6. Ordenação dos resultados por k

In [ ]:
results_knn = []

for result in knn.cv_results_:
    model = result["params"]["model"]

    results_knn.append({
        "k": model.n_neighbors, 
        "RMSE": result["mean_score"]
    })

results_knn = sorted(
    results_knn,
    key=lambda x: x["k"]
)

for result in results_knn:
    print(
        f"k = {result["k"]:3d} |"
        f"RMSE = {result["RMSE"]:.4f}"
    )

### 7. Gráfico RMSE x k

In [ ]:
ks = [r["k"] for r in results_knn]
rmses = [r["RMSE"] for r in results_knn]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    ks, 
    rmses,
    marker="o",
    linewidth=2,
    markersize=7
)

ax.scatter(
    best_k,
    best_rmse,
    s=100,
    marker="*",
    label=f"Melhor k = {best_k}"
)

ax.set_title("KNN - RMSE em função do número de vizinhos")
ax.set_xlabel("Número de vizinhos (k)")
ax.set_ylabel("RMSE médio na validação cruzada")

ax.set_xticks(ks)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()

save_figure(fig, "enem_knn_rmse")

plt.show()

### 8. Treinamento do modelo final

In [ ]:
final_model  = ScaledRegressor(
    model = KNeighborsRegressor(n_neighbors=best_k),
    block_size=1000    
)

final_model.fit(X_train, y_train)

print("Modelo KNN ajustado no treino completo.")
print("Melhor k:", best_k)

In [ ]:
save_table(
    results_knn, 
    "enem_knn_cv"
)